# Train last-token directions on all model-specific AIT data

This Colab notebook trains mean difference, logistic regression, and one-dimensional DAS directions from **last-token activations** using every available example in each model-specific AIT split. It runs GPT-2 Small, Qwen3-0.6B Base, Gemma 2B, and Pythia 1.4B with their tokenizer-matched pair configurations.

Original AIT train data fits directions, original validation data selects DAS epoch checkpoints and the best layer for every method, and original test data remains locked until final evaluation. Checkpoints, direction metadata, per-case records, summary CSVs, and figures are written directly to a timestamped Google Drive run.

## Before running

1. Choose a Colab GPU runtime. This full four-model, all-layer DAS sweep is substantially larger than the sampled experiment; an A100 is strongly preferable.
2. Have a Hugging Face token with read access to the private AIT dataset and any gated model repositories.
3. Leave `RESUME_RUN_ID = None` for a new run. After a disconnection, set it to the existing run directory name to resume compatible checkpoints.
4. For a reportable run, pin `PROJECT_REVISION` and every model revision in the YAML. Resolved model and tokenizer revisions are still recorded in the artifacts.

## 1. User settings

In [1]:
PROJECT_URL = "https://github.com/Adefioye/sentiment-manifold.git"
PROJECT_REVISION = None  # Set an immutable commit or tag for a reportable run.

DRIVE_STORAGE_ROOT = "/content/drive/MyDrive/sentiment-geometry"
TIMEZONE_NAME = "America/Chicago"
RESUME_RUN_ID = None  # Example: "2026-09-23_01-53_CDT"

DEVICE = "cuda"
DTYPE = "auto"
MODEL_NAMES = [
    "gpt2-small",
    "qwen-0.6b",
    "gemma-2b",
    "pythia-1.4b",
]
METHODS = ["mean_diff", "logistic_regression", "das"]
RUN_EXPERIMENT = True

## 2. Install and verify the project

The notebook delegates dataset loading, activation extraction, fitting, causal evaluation, similarity calculation, and persistence to package APIs.

In [2]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/sentiment-manifold")
if not (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", PROJECT_URL, str(PROJECT_ROOT)], check=True)
else:
    print(f"Reusing {PROJECT_ROOT}")

project_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if PROJECT_REVISION is not None:
    expected_commit = subprocess.check_output(
        ["git", "rev-parse", PROJECT_REVISION], cwd=PROJECT_ROOT, text=True
    ).strip()
    if project_commit != expected_commit:
        raise RuntimeError(
            f"Existing checkout is {project_commit}, expected {expected_commit}. "
            "Use a fresh runtime or update PROJECT_REVISION."
        )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[notebooks]"],
    check=True,
)
os.chdir(PROJECT_ROOT)
project_root_string = str(PROJECT_ROOT.resolve())
if project_root_string in sys.path:
    sys.path.remove(project_root_string)
sys.path.insert(0, project_root_string)
for loaded_name in list(sys.modules):
    if loaded_name == "sentiment_geometry" or loaded_name.startswith("sentiment_geometry."):
        del sys.modules[loaded_name]
importlib.invalidate_caches()

import sentiment_geometry
imported_root = Path(sentiment_geometry.__file__).resolve().parent
expected_root = (PROJECT_ROOT / "sentiment_geometry").resolve()
if imported_root != expected_root:
    raise ImportError(f"Imported {imported_root}, expected {expected_root}")
print("Project commit:", project_commit)
print("Package root:  ", imported_root)

Project commit: f0a8d4a599c8f2c01b03d2caa1f763317bce31d9
Package root:   /content/sentiment-manifold/sentiment_geometry


## 3. Mount Google Drive and create the run

In [3]:
import torch

from sentiment_geometry.persistence import maybe_mount_google_drive, prepare_timestamped_run

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Enable a GPU runtime before continuing.")

maybe_mount_google_drive(True)
if "RUN_LAYOUT" not in globals() or RESUME_RUN_ID is not None:
    RUN_LAYOUT = prepare_timestamped_run(
        DRIVE_STORAGE_ROOT,
        experiment_name="full-ait-last-token-directions",
        timezone_name=TIMEZONE_NAME,
        resume_run_id=RESUME_RUN_ID,
    )

print("Run ID:             ", RUN_LAYOUT.run_id)
print("Run root:           ", RUN_LAYOUT.root)
print("Result CSVs:        ", RUN_LAYOUT.results_dir)
print("Direction artifacts:", RUN_LAYOUT.directions_dir)
print("Figures:            ", RUN_LAYOUT.figures_dir)
if torch.cuda.is_available():
    print("GPU:                ", torch.cuda.get_device_name(0))

Mounted at /content/drive
Run ID:              2026-09-23_09-10_CDT
Run root:            /content/drive/MyDrive/sentiment-geometry/full-ait-last-token-directions/runs/2026-09-23_09-10_CDT
Result CSVs:         /content/drive/MyDrive/sentiment-geometry/full-ait-last-token-directions/runs/2026-09-23_09-10_CDT/results
Direction artifacts: /content/drive/MyDrive/sentiment-geometry/full-ait-last-token-directions/runs/2026-09-23_09-10_CDT/directions
Figures:             /content/drive/MyDrive/sentiment-geometry/full-ait-last-token-directions/runs/2026-09-23_09-10_CDT/figures
GPU:                 NVIDIA L4


## 4. Authenticate to Hugging Face

The token is entered through a hidden prompt, passed only to the training subprocess, and cleared immediately afterward.

In [4]:
import gc
from getpass import getpass

from huggingface_hub import HfApi
from huggingface_hub.utils import reset_sessions

_RUNTIME_SECRETS = {}

def get_runtime_secret(name):
    if name not in _RUNTIME_SECRETS:
        value = getpass(f"Enter {name} (input hidden): " ).strip()
        if not value:
            raise RuntimeError(f"{name} was not provided.")
        _RUNTIME_SECRETS[name] = value
    return _RUNTIME_SECRETS[name]

def clear_hf_credentials():
    os.environ.pop("HF_TOKEN", None)
    value = _RUNTIME_SECRETS.pop("HF_TOKEN", None)
    if value is not None:
        del value
    reset_sessions()
    gc.collect()

_token = get_runtime_secret("HF_TOKEN")
try:
    hf_account = HfApi(token=_token).whoami()["name"]
finally:
    del _token
print(f"Authenticated to Hugging Face as {hf_account}. Token value was not displayed.")

Authenticated to Hugging Face as kokolamba. Token value was not displayed.


## 5. Inspect and lock the full-data contract

The three `null` sampling caps mean that the loader consumes every available matched example from each model-specific source split; the data are not pooled or repartitioned.

In [5]:
from dataclasses import asdict

import pandas as pd
from IPython.display import display

from sentiment_geometry.experiments import AITValenceExperimentConfig
from sentiment_geometry.persistence import RunArtifactStore

CONFIG_PATH = PROJECT_ROOT / "configs/full_ait_valence_directions.yaml"
base_config = AITValenceExperimentConfig.load(CONFIG_PATH)
configured_model_names = [model.name for model in base_config.models]
if MODEL_NAMES != configured_model_names:
    raise RuntimeError(
        f"Notebook models {MODEL_NAMES} do not match full-AIT config models "
        f"{configured_model_names}."
    )
model_pair_configs = {
    name: base_config.data.matched_config_for(name) for name in MODEL_NAMES
}
expected_pair_configs = {
    "gpt2-small": "gpt2_small_matched_pairs",
    "qwen-0.6b": "qwen_0_6b_matched_pairs",
    "gemma-2b": "gemma_2b_matched_pairs",
    "pythia-1.4b": "pythia_1_4b_matched_pairs",
}
if model_pair_configs != expected_pair_configs:
    raise RuntimeError(
        f"Unexpected model-specific AIT pair configurations: {model_pair_configs}"
    )
if base_config.data.revision is None:
    raise RuntimeError("The private AIT dataset revision must be pinned.")
if not base_config.sampling.uses_all_available:
    raise RuntimeError("Full AIT training requires all three sampling caps to be null.")
if base_config.sweep.activation_representation != "last_token":
    raise RuntimeError("The full-AIT config must use last-token activations.")
if base_config.selection.das_checkpoint_split != "eval":
    raise RuntimeError("DAS checkpoints must be selected on AIT validation data.")
if base_config.selection.layer_selection_split != "eval":
    raise RuntimeError("All method layers must be selected on AIT validation data.")
if base_config.selection.final_evaluation_split != "test":
    raise RuntimeError("AIT test data must be reserved for final evaluation.")

contract = {
    "data_scope": "all_available_model_specific_ait",
    "activation_representation": "last_token",
    "fit_position": "final",
    "methods": METHODS,
    "layers": "all_non_embedding",
    "sampling_caps": {"train": None, "validation": None, "test": None},
    "das_checkpoint_role": "eval",
    "layer_selection_role": "eval",
    "layer_selection_metric": "logit_flip_percent",
    "final_evaluation_role": "test",
    "model_matched_configs": model_pair_configs,
    "test_is_unbiased_final_evaluation": True,
}
RunArtifactStore(RUN_LAYOUT.root).write_json("notebook_contract.json", contract)
display(pd.DataFrame([asdict(model) for model in base_config.models]))
display(pd.Series(contract, name="value").to_frame())

,name,revision,prepend_bos,device,dtype,batch_size
0,gpt2-small,607a30d783dfa663caf39e06633721c8d4cfcd7e,True,auto,auto,16
1,qwen-0.6b,da87bfb608c14b7cf20ba1ce41287e8de496c0cd,False,auto,auto,8
2,gemma-2b,None,True,auto,auto,4
3,pythia-1.4b,None,True,auto,auto,8


,value
data_scope,all_available_model_specific_ait
activation_representation,last_token
fit_position,final
methods,"[mean_diff, logistic_regression, das]"
layers,all_non_embedding
sampling_caps,"{'train': None, 'validation': None, 'test': None}"
das_checkpoint_role,eval
layer_selection_role,eval
layer_selection_metric,logit_flip_percent
final_evaluation_role,test


## 6. Record the software environment

In [6]:
import platform
from importlib.metadata import version

environment = {
    "project_commit": project_commit,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": version("torch"),
    "transformers": version("transformers"),
    "datasets": version("datasets"),
    "numpy": version("numpy"),
    "pandas": version("pandas"),
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
RunArtifactStore(RUN_LAYOUT.root).write_json("environment.json", environment)
display(pd.Series(environment, name="value").to_frame())

,value
project_commit,f0a8d4a599c8f2c01b03d2caa1f763317bce31d9
python,3.13.15
platform,Linux-6.6.122+-x86_64-with-glibc2.39
torch,2.11.0+cu128
transformers,4.57.6
datasets,4.8.5
numpy,2.1.3
pandas,2.2.3
device,cuda
gpu,NVIDIA L4


## 7. Train on all model-specific AIT data

This cell invokes the real `--last-token` and `--all-non-embedding-layers` CLI flags. Combined stdout and stderr are streamed to the notebook and saved to `training.log`. Compatible layer checkpoints are resumed automatically.

In [7]:
import shlex
import shutil

cli_path = shutil.which("sentiment-geometry")
if cli_path is None:
    raise RuntimeError("The sentiment-geometry CLI was not installed.")

command = [
    cli_path,
    "train-ait-valence",
    "--config", str(CONFIG_PATH),
    "--device", DEVICE,
    "--dtype", DTYPE,
    "--output-dir", str(RUN_LAYOUT.results_dir),
    "--checkpoint-dir", str(RUN_LAYOUT.directions_dir),
    "--hf-token-env", "HF_TOKEN",
    "--last-token",
    "--all-non-embedding-layers",
]
for model_name in MODEL_NAMES:
    command.extend(["--model", model_name])
for method in METHODS:
    command.extend(["--method", method])

training_log_path = RUN_LAYOUT.root / "training.log"

def run_logged(command, *, cwd, env, log_path):
    command = [str(part) for part in command]
    command_display = shlex.join(command)
    header = f"\n$ {command_display}\n"
    print(header, end="")
    with Path(log_path).open("a", encoding="utf-8", buffering=1) as log:
        log.write(header)
        process = subprocess.Popen(
            command,
            cwd=cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
            returncode = process.wait()
        except BaseException:
            if process.poll() is None:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise
    if returncode != 0:
        print(f"Training failed; full combined output is in {log_path}")
        raise subprocess.CalledProcessError(returncode, command)
    return subprocess.CompletedProcess(command, returncode)

if RUN_EXPERIMENT:
    child_environment = os.environ.copy()
    child_environment["HF_TOKEN"] = get_runtime_secret("HF_TOKEN")
    child_environment["PYTHONUNBUFFERED"] = "1"
    RUN_LAYOUT.update_manifest(
        status="running",
        metadata={
            "project_commit": project_commit,
            "representation": "last_token",
            "data_scope": "all_available_model_specific_ait",
        },
    )
    try:
        run_logged(
            command,
            cwd=PROJECT_ROOT,
            env=child_environment,
            log_path=training_log_path,
        )
        RUN_LAYOUT.update_manifest(status="trained")
    except BaseException as error:
        failure_metadata = {
            "failure_type": type(error).__name__,
            "training_log": str(training_log_path),
        }
        if isinstance(error, subprocess.CalledProcessError):
            failure_metadata["exit_code"] = error.returncode
        RUN_LAYOUT.update_manifest(status="failed", metadata=failure_metadata)
        raise
    finally:
        child_environment.pop("HF_TOKEN", None)
        del child_environment
        clear_hf_credentials()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    clear_hf_credentials()
    print("RUN_EXPERIMENT is False; loading an existing run only.")


$ /usr/local/bin/sentiment-geometry train-ait-valence --config /content/sentiment-manifold/configs/full_ait_valence_directions.yaml --device cuda --dtype auto --output-dir /content/drive/MyDrive/sentiment-geometry/full-ait-last-token-directions/runs/2026-09-23_09-10_CDT/results --checkpoint-dir /content/drive/MyDrive/sentiment-geometry/full-ait-last-token-directions/runs/2026-09-23_09-10_CDT/directions --hf-token-env HF_TOKEN --last-token --all-non-embedding-layers --model gpt2-small --model qwen-0.6b --model gemma-2b --model pythia-1.4b --method mean_diff --method logistic_regression --method das

Fetching 1 files: 100%|██████████| 1/1 [00:01<00:00,  1.79s/it]

Generating train split: 0 examples [00:00, ? examples/s]
Generating train split: 321 examples [00:00, 18274.72 examples/s]

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00,  1.07it/s]

Generating validation split: 0 examples [00:00, ? examples/s]
Generating validation split: 99 examples [00:00, 23383.04 examples/s]

Fetchi

KeyboardInterrupt: 

## 8. Validate and display the data, selections, and locked-test metrics

In [ ]:
RESULTS_DIR = RUN_LAYOUT.results_dir
dataset_summary = pd.read_csv(RESULTS_DIR / "dataset_summary.csv")
selection = pd.read_csv(RESULTS_DIR / "all_models_layer_selection.csv")
final_metrics = pd.read_csv(RESULTS_DIR / "all_models_final_metrics.csv")
direction_metadata = pd.read_csv(RESULTS_DIR / "all_models_direction_metadata.csv")

observed_pair_configs = dict(
    dataset_summary.groupby("model")["dataset_config"].first()
)
if observed_pair_configs != model_pair_configs:
    raise RuntimeError(
        f"Unexpected result pair configurations: {observed_pair_configs}"
    )
if not set(dataset_summary["source_split"]) == {"train", "validation", "test"}:
    raise RuntimeError("The full AIT run did not preserve all three source splits.")
expected_cells = len(MODEL_NAMES) * len(METHODS)
if len(selection) != expected_cells:
    raise RuntimeError(f"Expected {expected_cells} selected layers; found {len(selection)}")
if set(selection["selection_dataset"]) != {"ait_eval"}:
    raise RuntimeError("A layer was not selected on AIT validation data.")
if set(final_metrics["dataset"]) != {"ait_test"}:
    raise RuntimeError("Final metrics were not evaluated on locked AIT test data.")
if set(final_metrics["phase"]) != {"final_evaluation"}:
    raise RuntimeError("AIT test metrics are not marked as final evaluation.")
required_test_metrics = {"logit_flip_percent", "sign_flip_percent"}
missing_test_metrics = required_test_metrics - set(final_metrics.columns)
if missing_test_metrics:
    raise RuntimeError(f"Locked-test metrics are missing: {sorted(missing_test_metrics)}")

split_order = [
    base_config.data.train_split,
    base_config.data.eval_split,
    base_config.data.test_split,
]
if (dataset_summary["n_directed_cases"] % 2 != 0).any():
    raise RuntimeError("Every AIT matched pair must produce two directed cases.")
dataset_summary["n_matched_pairs"] = dataset_summary["n_directed_cases"] // 2
matched_pair_count_table = dataset_summary.pivot(
    index="model", columns="source_split", values="n_matched_pairs"
).reindex(columns=split_order)
matched_pair_count_table["total"] = matched_pair_count_table.sum(axis=1)
example_count_table = dataset_summary.pivot(
    index="model", columns="source_split", values="n_examples"
).reindex(columns=split_order)
example_count_table["total"] = example_count_table.sum(axis=1)
directed_case_count_table = dataset_summary.pivot(
    index="model", columns="source_split", values="n_directed_cases"
).reindex(columns=split_order)
directed_case_count_table["total"] = directed_case_count_table.sum(axis=1)
layer_table = selection.pivot(index="model", columns="method", values="selected_layer")
validation_logit_flip_table = selection.pivot(
    index="model", columns="method", values="selection_value_percent"
).round(3)
test_logit_flip_table = final_metrics.pivot(
    index="model", columns="method", values="logit_flip_percent"
).round(3)
test_sign_flip_table = final_metrics.pivot(
    index="model", columns="method", values="sign_flip_percent"
).round(3)
selected_checkpoint_table = direction_metadata[
    direction_metadata["selected_layer"].astype(bool)
]["model method layer selected_epoch artifact_path".split()]

display(matched_pair_count_table.style.set_caption("All available model-specific AIT matched pairs"))
display(example_count_table.style.set_caption("All available model-specific AIT examples"))
display(directed_case_count_table.style.set_caption("All available AIT directed cases"))
display(layer_table.style.set_caption("Validation-selected residual boundary"))
display(validation_logit_flip_table.style.set_caption("Validation layer-selection logit flip percent"))
display(test_logit_flip_table.style.set_caption("Locked-test logit flip percent"))
display(test_sign_flip_table.style.set_caption("Locked-test sign flip percent"))
display(selected_checkpoint_table.style.set_caption("Selected direction checkpoints"))

## 9. First, middle, and last boundary direction similarity

Absolute cosine is displayed because it compares direction axes without treating an orientation reversal as a different subspace. Signed cosine remains in the saved CSV for orientation auditing.

In [ ]:
similarities = pd.read_csv(RESULTS_DIR / "all_models_direction_similarities.csv")
method_rank = {method: index for index, method in enumerate(METHODS)}
similarities["method_a_rank"] = similarities["method_a"].map(method_rank)
similarities["method_b_rank"] = similarities["method_b"].map(method_rank)
similarity_summary = similarities[
    similarities["method_a_rank"] < similarities["method_b_rank"]
].copy()

role_rows = []
for model_name, model_rows in similarity_summary.groupby("model", sort=False):
    boundaries = sorted(model_rows["layer"].unique())
    if len(boundaries) != 3:
        raise RuntimeError(
            f"Expected first/middle/last boundaries for {model_name}; got {boundaries}"
        )
    roles = dict(zip(boundaries, ["first", "middle", "last"]))
    selected = model_rows.copy()
    selected["boundary_role"] = selected["layer"].map(roles)
    role_rows.append(selected)
similarity_summary = pd.concat(role_rows, ignore_index=True)
similarity_summary["method_pair"] = (
    similarity_summary["method_a"] + " vs " + similarity_summary["method_b"]
)
summary_columns = [
    "model", "boundary_role", "layer", "method_a", "method_b",
    "signed_cosine", "absolute_cosine",
]
summary_path = RESULTS_DIR / "snapshot_similarity_summary.csv"
similarity_summary[summary_columns].to_csv(summary_path, index=False)
similarity_table = similarity_summary.pivot(
    index=["model", "boundary_role", "layer"],
    columns="method_pair",
    values="absolute_cosine",
).round(3)
display(similarity_table.style.set_caption("Absolute cosine at first/middle/last boundaries"))
print("Saved compact similarity report:", summary_path)

## 10. Save and display plots

The reporting API saves layer curves, selected-layer metrics, and one cosine-similarity heatmap for the first, middle, and last residual boundary of every model.

In [ ]:
from IPython.display import Image, display

from sentiment_geometry.reporting import plot_ait_valence_run

figure_paths = plot_ait_valence_run(
    RESULTS_DIR, figure_dir=RUN_LAYOUT.figures_dir
)
figure_manifest = pd.DataFrame(
    {
        "figure": [path.name for path in figure_paths],
        "path": [str(path) for path in figure_paths],
    }
)
figure_manifest.to_csv(RESULTS_DIR / "figure_manifest.csv", index=False)
display(figure_manifest)
for path in figure_paths:
    display(Image(filename=str(path)))

## 11. Audit saved artifacts and clear credentials

In [ ]:
required_result_files = [
    "requested_config.json",
    "sample_manifest.csv",
    "pair_manifest.csv",
    "dataset_summary.csv",
    "all_models_metrics.csv",
    "all_models_patching_records.csv",
    "all_models_direction_metadata.csv",
    "all_models_das_epoch_metrics.csv",
    "all_models_direction_similarities.csv",
    "all_models_layer_selection.csv",
    "all_models_selected_metrics.csv",
    "all_models_final_metrics.csv",
    "all_models_final_patching_records.csv",
    "snapshot_similarity_summary.csv",
    "figure_manifest.csv",
    "experiment_manifest.json",
]
missing = [name for name in required_result_files if not (RESULTS_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing required artifacts: {missing}")

required_model_files = [
    "sample_manifest.csv", "pair_manifest.csv", "dataset_summary.csv",
    "metrics.csv", "patching_records.csv", "direction_metadata.csv",
    "das_epoch_metrics.csv", "direction_similarities.csv",
    "layer_selection.csv", "selected_metrics.csv",
    "final_metrics.csv", "final_patching_records.csv",
]
for model_name in MODEL_NAMES:
    model_result_dir = RESULTS_DIR / model_name
    missing_model_files = [
        name for name in required_model_files
        if not (model_result_dir / name).is_file()
    ]
    if missing_model_files:
        raise FileNotFoundError(
            f"Missing {model_name} result artifacts: {missing_model_files}"
        )

checkpoint_files = list(RUN_LAYOUT.directions_dir.rglob("*.npz"))
if len(checkpoint_files) < len(direction_metadata):
    raise RuntimeError(
        f"Expected at least {len(direction_metadata)} checkpoints; "
        f"found {len(checkpoint_files)}"
    )
if os.environ.get("HF_TOKEN") is not None:
    raise RuntimeError("HF_TOKEN remains in the notebook environment.")
if "HF_TOKEN" in _RUNTIME_SECRETS:
    raise RuntimeError("HF_TOKEN remains in the notebook secret cache.")

RUN_LAYOUT.update_manifest(
    status="completed",
    metadata={
        "models": MODEL_NAMES,
        "methods": METHODS,
        "representation": "last_token",
        "data_scope": "all_available_model_specific_ait",
        "direction_checkpoints": len(checkpoint_files),
        "direction_rows": len(direction_metadata),
        "similarity_rows": len(similarities),
        "figures": len(figure_paths),
    },
)
print("Run completed and audited:", RUN_LAYOUT.root)
print("Verified: HF_TOKEN is absent from the environment and notebook secret cache.")
_RUNTIME_SECRETS.clear()